# Demo: Klasifikasi Lalu Lintas Jaringan DDoS
**LSTM vs GRU vs Random Forest pada CIC-DDoS2019**

Kelompok 08  -  Mata Kuliah Kecerdasan Buatan, Teknik Komputer UI

---
Notebook ini mendemonstrasikan tiga skenario:
1. **Perbandingan fitur**  -  distribusi nilai fitur normal vs DDoS
2. **Prediksi pada sampel nyata**  -  5 normal + 5 DDoS dari test set
3. **Traffic sintetis buatan**  -  trafik UDP flood, HTTP normal, kasus borderline

In [ ]:
# ── 0. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 1. Paths  -  sesuaikan jika berbeda ─────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/tugas-akhir-ai'
SPLITS_DIR = f'{DRIVE_ROOT}/splits'
LSTM_CKPT  = f'{DRIVE_ROOT}/best_lstm.pt'
GRU_CKPT   = f'{DRIVE_ROOT}/best_gru.pt'
RF_CKPT    = f'{DRIVE_ROOT}/best_rf.pkl'

import os
for p in [SPLITS_DIR, LSTM_CKPT, GRU_CKPT, RF_CKPT]:
    status = '✓' if os.path.exists(p) else '✗ NOT FOUND'
    print(f'  {status}  {p}')

In [ ]:
# -- 2b. Clone repo & load config.yaml ------------------------------------
import subprocess, sys, yaml
import os

REPO_URL = "https://github.com/calvinkatoroy/tugas-akhir-ai-kel-06.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

sys.path.insert(0, f"{REPO_DIR}/src")

with open(f"{REPO_DIR}/config.yaml") as f:
    cfg = yaml.safe_load(f)

FEATURES   = cfg["features"]
N_FEATURES = len(FEATURES)
SEQ_LEN    = cfg["lstm"]["seq_len"]
LSTM_CFG   = cfg["lstm"]   # hidden_size, num_layers, dropout, etc.
GRU_CFG    = cfg["gru"]

print(f"Config loaded from {REPO_DIR}/config.yaml")
print(f"Features ({N_FEATURES}): {FEATURES}")
print(f"SEQ_LEN: {SEQ_LEN}")
print(f"LSTM: hidden={LSTM_CFG["hidden_size"]}, layers={LSTM_CFG["num_layers"]}, dropout={LSTM_CFG["dropout"]}")
print(f"GRU:  hidden={GRU_CFG["hidden_size"]}, layers={GRU_CFG["num_layers"]}, dropout={GRU_CFG["dropout"]}")


In [ ]:
# -- 2. Imports & seed -------------------------------------------------------
import random, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLR    = {0: "#2196F3", 1: "#F44336"}  # blue=normal, red=ddos
LABEL  = {0: "NORMAL", 1: "DDoS"}

print(f"Device: {DEVICE}  |  fitur & seq_len akan di-load dari config.yaml")


In [ ]:
# ── 3. Model definitions (inline  -  tidak perlu import dari src/) ───────────
class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0,
                            batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])

class GRUClassifier(nn.Module):
    def __init__(self, n_features, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden_size, num_layers=num_layers,
                          dropout=dropout if num_layers > 1 else 0.0,
                          batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        out, _ = self.gru(x)
        return self.classifier(out[:, -1, :])

print('Model classes defined.')

In [ ]:
# ── 4. Load models & test data ────────────────────────────────────────────
# LSTM
lstm = LSTMClassifier(
    N_FEATURES,
    hidden_size=LSTM_CFG["hidden_size"],
    num_layers=LSTM_CFG["num_layers"],
    dropout=LSTM_CFG["dropout"]
).to(DEVICE)
lstm.load_state_dict(torch.load(LSTM_CKPT, map_location=DEVICE))
lstm.eval()

# GRU
gru = GRUClassifier(
    N_FEATURES,
    hidden_size=GRU_CFG["hidden_size"],
    num_layers=GRU_CFG["num_layers"],
    dropout=GRU_CFG["dropout"]
).to(DEVICE)
gru.load_state_dict(torch.load(GRU_CKPT, map_location=DEVICE))
gru.eval()

# Random Forest
rf = joblib.load(RF_CKPT)

# Scaler (untuk synthetic demo)
scaler = joblib.load(f'{SPLITS_DIR}/scaler.pkl')

# Test data
X_test_seq = np.load(f'{SPLITS_DIR}/X_test_seq.npy')  # (n, 10, 16)
y_test_seq = np.load(f'{SPLITS_DIR}/y_test_seq.npy')  # (n,)
X_test     = np.load(f'{SPLITS_DIR}/X_test.npy')      # (n, 16)  -  for RF
y_test     = np.load(f'{SPLITS_DIR}/y_test.npy')      # (n,)

print(f'LSTM loaded   -  params: {sum(p.numel() for p in lstm.parameters()):,}')
print(f'GRU  loaded   -  params: {sum(p.numel() for p in gru.parameters()):,}')
print(f'RF   loaded   -  estimators: {rf.n_estimators}')
print(f'Test (seq):  {X_test_seq.shape}  |  Test (flat): {X_test.shape}')
print(f'Class dist test (seq): normal={int((y_test_seq==0).sum()):,}  ddos={int((y_test_seq==1).sum()):,}')

---
## Demo 1  -  Perbandingan Fitur: Normal vs DDoS
Sebelum melihat prediksi, mari lihat perbedaan statistik fitur antara kedua kelas.

In [ ]:
# Gunakan flat test set  -  unscale untuk tampilan yang readable
X_inv = scaler.inverse_transform(X_test)
df_test = pd.DataFrame(X_inv, columns=FEATURES)
df_test['label'] = y_test

# Fitur yang paling informatif untuk demo
KEY_FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Flow Packets/s', 'Flow Bytes/s', 'Fwd IAT Mean',
    'SYN Flag Count', 'ACK Flag Count',
]

stats = df_test.groupby('label')[KEY_FEATURES].median().T
stats.columns = ['NORMAL (median)', 'DDoS (median)']
stats['Rasio DDoS/Normal'] = (stats['DDoS (median)'] / stats['NORMAL (median)'].replace(0, 1)).round(1)
print('=== Statistik Median Fitur per Kelas (nilai asli, belum di-scale) ===')
print(stats.to_string())

In [ ]:
# Visualisasi 4 fitur kunci
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
demo_feats = ['Flow Packets/s', 'Flow Duration', 'ACK Flag Count', 'Fwd IAT Mean']

for ax, feat in zip(axes, demo_feats):
    for lbl in [0, 1]:
        vals = df_test[df_test['label'] == lbl][feat]
        # clip untuk visualisasi (hindari outlier ekstrem)
        clip_hi = vals.quantile(0.95)
        vals = vals.clip(upper=clip_hi)
        ax.hist(vals, bins=50, alpha=0.6, color=CLR[lbl], label=LABEL[lbl], density=True)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_xlabel('Nilai (raw)')
    ax.set_ylabel('Densitas')
    ax.legend(fontsize=8)

plt.suptitle('Distribusi Fitur Kunci: Normal vs DDoS (95th percentile clip)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Demo 2  -  Prediksi pada Sampel Nyata dari Test Set
Pilih 5 sampel normal + 5 sampel DDoS dari test set, jalankan ketiga model.

In [ ]:
# Pilih sampel (index ke-y dari test_seq)
rng = np.random.default_rng(SEED)
idx_normal = rng.choice(np.where(y_test_seq == 0)[0], size=5, replace=False)
idx_ddos   = rng.choice(np.where(y_test_seq == 1)[0], size=5, replace=False)
idx_demo   = np.concatenate([idx_normal, idx_ddos])
y_demo     = y_test_seq[idx_demo]

# Slice sequences untuk LSTM/GRU
X_demo_seq = X_test_seq[idx_demo]              # (10, 10, 16)

# Untuk RF: gunakan timestep terakhir dari sequence (sudah di-scale)
X_demo_flat = X_demo_seq[:, -1, :]             # (10, 16)

print(f'Sampel demo: {len(idx_demo)} sampel')
print(f'  Index normal : {idx_normal}')
print(f'  Index DDoS   : {idx_ddos}')

In [ ]:
# Tampilkan nilai fitur (unscaled) untuk sampel demo
X_demo_unscaled = scaler.inverse_transform(X_demo_flat)
df_demo = pd.DataFrame(X_demo_unscaled, columns=FEATURES)
df_demo.insert(0, 'Label Asli', [LABEL[y] for y in y_demo])
df_demo.index = [f'S{i+1}' for i in range(len(idx_demo))]

display_cols = ['Label Asli', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
                'Flow Packets/s', 'Flow Bytes/s', 'Fwd IAT Mean', 'ACK Flag Count', 'SYN Flag Count']
print('=== Nilai Fitur Sampel Demo (nilai asli, pembulatan 1 desimal) ===')
print(df_demo[display_cols].round(1).to_string())

In [ ]:
# ── Jalankan inferensi ketiga model ───────────────────────────────────────
def predict_all(X_seq, X_flat):
    """Return dict of {model_name: (pred_label, prob_ddos)}"""
    results = {}

    # LSTM
    with torch.no_grad():
        t = torch.tensor(X_seq, dtype=torch.float32).to(DEVICE)
        logits = lstm(t)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
    results['LSTM'] = (probs.argmax(1), probs[:, 1])

    # GRU
    with torch.no_grad():
        logits = gru(t)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
    results['GRU'] = (probs.argmax(1), probs[:, 1])

    # RF
    rf_probs = rf.predict_proba(X_flat)
    results['RF'] = (rf_probs.argmax(1), rf_probs[:, 1])

    return results

results = predict_all(X_demo_seq, X_demo_flat)

# Tampilkan hasil
rows = []
for i, (y_true) in enumerate(y_demo):
    row = {'Sampel': f'S{i+1}', 'Label Asli': LABEL[y_true]}
    for model_name, (preds, probs) in results.items():
        pred_lbl = LABEL[preds[i]]
        prob_ddos = probs[i]
        benar = '✓' if preds[i] == y_true else '✗'
        row[f'{model_name} Pred'] = f'{pred_lbl} ({prob_ddos:.1%}) {benar}'
    rows.append(row)

df_results = pd.DataFrame(rows).set_index('Sampel')
print('=== Hasil Prediksi Ketiga Model ===')
print(df_results.to_string())

In [ ]:
# Visualisasi: heatmap probabilitas DDoS
prob_mat = np.array([results[m][1] for m in ['LSTM', 'GRU', 'RF']])  # (3, 10)
sample_labels = [f"S{i+1}\n({LABEL[y_demo[i]]})".replace('DDoS','DDoS') for i in range(len(y_demo))]

fig, ax = plt.subplots(figsize=(13, 3))
im = ax.imshow(prob_mat, aspect='auto', vmin=0, vmax=1, cmap='RdYlGn_r')
ax.set_xticks(range(len(y_demo)))
ax.set_xticklabels(sample_labels, fontsize=9)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['LSTM', 'GRU', 'RF'], fontweight='bold')
ax.set_title('Probabilitas DDoS per Sampel (merah = yakin DDoS, hijau = yakin normal)', fontweight='bold')
plt.colorbar(im, ax=ax, label='P(DDoS)')

for i in range(3):
    for j in range(len(y_demo)):
        ax.text(j, i, f'{prob_mat[i, j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if prob_mat[i, j] > 0.5 else 'black')

plt.tight_layout()
plt.show()

---
## Demo 3  -  Trafik Sintetis Buatan
Kita buat 3 skenario trafik secara manual:
- **Skenario A**: UDP Flood (serangan volumetrik jelas)
- **Skenario B**: HTTP normal (browsing biasa)
- **Skenario C**: Borderline (burst CDN  -  bisa terlihat seperti DDoS)

Nilai dimasukkan dalam satuan **asli** (sebelum scaling), lalu di-scale otomatis.

In [ ]:
# Definisi trafik sintetis (nilai RAW/asli  -  sebelum StandardScaler)
# Urutan: sesuai FEATURES list di atas
#          [FlowDur, FwdPkt, BwdPkt, FwdPktLen, BwdPktLen,
#           FwdPktMean, BwdPktMean, FlowBytes, FlowPkts,
#           FwdIAT, BwdIAT, SYN, RST, PSH, ACK, AvgPktSize]

SYNTHETIC = {
    'A - UDP Flood': [
        42,         # Flow Duration (microsec)  -  sangat pendek
        1,          # Total Fwd Packets
        0,          # Total Backward Packets  -  server tidak balas
        52,         # Fwd Packets Length Total
        0,          # Bwd Packets Length Total
        52.0,       # Fwd Packet Length Mean
        0.0,        # Bwd Packet Length Mean
        1_428_571,  # Flow Bytes/s  -  sangat tinggi
        23_810,     # Flow Packets/s  -  sangat tinggi
        0.0,        # Fwd IAT Mean (hanya 1 paket)
        0.0,        # Bwd IAT Mean
        0,          # SYN Flag
        0,          # RST Flag
        0,          # PSH Flag
        0,          # ACK Flag
        52.0,       # Avg Packet Size
    ],
    'B - HTTP Normal': [
        4_500_000,  # Flow Duration (4.5 detik)
        18,         # Total Fwd Packets
        14,         # Total Backward Packets
        8_200,      # Fwd Packets Length Total
        45_000,     # Bwd Packets Length Total
        455.6,      # Fwd Packet Length Mean
        3_214.3,    # Bwd Packet Length Mean
        11_822,     # Flow Bytes/s
        7.1,        # Flow Packets/s
        282_000,    # Fwd IAT Mean
        375_000,    # Bwd IAT Mean
        1,          # SYN Flag
        0,          # RST Flag
        4,          # PSH Flag
        13,         # ACK Flag
        1_678.6,    # Avg Packet Size
    ],
    'C - Borderline (CDN burst)': [
        120_000,    # Flow Duration (120ms)
        45,         # Total Fwd Packets
        38,         # Total Backward Packets  -  masih ada reply
        65_000,     # Fwd Packets Length Total
        90_000,     # Bwd Packets Length Total
        1_444.4,    # Fwd Packet Length Mean
        2_368.4,    # Bwd Packet Length Mean
        1_291_667,  # Flow Bytes/s  -  tinggi (CDN prefetch)
        691.7,      # Flow Packets/s  -  tinggi tapi tidak ekstrem
        2_727,      # Fwd IAT Mean  -  masih ada interval
        3_243,      # Bwd IAT Mean
        1,          # SYN Flag
        0,          # RST Flag
        8,          # PSH Flag
        38,         # ACK Flag
        1_818.2,    # Avg Packet Size
    ],
}

# Tampilkan fitur sintetis
df_syn = pd.DataFrame(SYNTHETIC, index=FEATURES).T
print('=== Nilai Fitur Trafik Sintetis (nilai asli) ===')
print(df_syn[['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
              'Flow Packets/s', 'Flow Bytes/s', 'Fwd IAT Mean', 'ACK Flag Count']].to_string())

In [ ]:
# Scale dan prediksi
scenario_names = list(SYNTHETIC.keys())
X_syn_raw  = np.array(list(SYNTHETIC.values()), dtype=np.float32)  # (3, 16)
X_syn_sc   = scaler.transform(X_syn_raw).astype(np.float32)         # scaled

# Untuk LSTM/GRU: repeat timestep 10x → (3, 10, 16) (steady-state)
X_syn_seq  = np.stack([X_syn_sc] * SEQ_LEN, axis=1)                 # (3, 10, 16)

syn_results = predict_all(X_syn_seq, X_syn_sc)

print('=== Prediksi Trafik Sintetis ===')
print(f'{"Skenario":<35} {"LSTM":<25} {"GRU":<25} {"RF":<25}')
print('-' * 110)
for i, name in enumerate(scenario_names):
    row = f'{name:<35}'
    for model_name in ['LSTM', 'GRU', 'RF']:
        pred = LABEL[syn_results[model_name][0][i]]
        prob = syn_results[model_name][1][i]
        row += f' {pred} ({prob:.1%}){" ":<10}'
    print(row)

In [ ]:
# Visualisasi: probability bar per skenario
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
model_names = ['LSTM', 'GRU', 'RF']
bar_clrs    = ['#9C27B0', '#FF9800', '#4CAF50']  # purple, orange, green

for ax, (name, raw_vals) in zip(axes, SYNTHETIC.items()):
    probs_ddos = [syn_results[m][1][list(SYNTHETIC.keys()).index(name)] for m in model_names]
    bars = ax.barh(model_names, probs_ddos, color=bar_clrs, height=0.5)
    ax.set_xlim(0, 1)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('P(DDoS)')
    ax.set_title(name, fontweight='bold', fontsize=10)

    for bar, p in zip(bars, probs_ddos):
        color = '#D32F2F' if p >= 0.5 else '#1976D2'
        ax.text(min(p + 0.02, 0.98), bar.get_y() + bar.get_height()/2,
                f'{p:.1%}', va='center', fontweight='bold', color=color, fontsize=10)

    verdict = 'DDoS' if np.mean(probs_ddos) >= 0.5 else 'NORMAL'
    bg_clr  = '#FFEBEE' if verdict == 'DDoS' else '#E3F2FD'
    ax.set_facecolor(bg_clr)

plt.suptitle('Keyakinan Model per Skenario (garis putus = ambang 50%)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Demo 4  -  Ringkasan Performa Model
Hasil evaluasi pada test set (64.705 sampel, 15% dari total dataset).

In [ ]:
# Tabel ringkasan performa final (dari hasil training yang sudah selesai)
summary = pd.DataFrame({
    'Model':     ['LSTM', 'GRU', 'Random Forest'],
    'Accuracy':  ['99.63%', '99.61%', '99.87%'],
    'Precision': ['99.87%', '99.92%', '99.96%'],
    'Recall':    ['99.65%', '99.58%', '99.88%'],
    'F1-Score':  ['0.9976', '0.9975', '0.9992'],
    'FPR':       ['0.46%',  '0.28%',  '0.15%'],
    'AUC-ROC':   ['0.9996', '0.9996', '0.9998'],
}).set_index('Model')
print('=== Perbandingan Akhir Ketiga Model (Test Set) ===')
print(summary.to_string())
print('\nCatatan: RF mengungguli LSTM/GRU karena fitur CICFlowMeter sudah merupakan')
print('statistik agregat aliran  -  informasi temporal sudah terabstraksi ke dalam fitur.')

In [ ]:
# Bar chart perbandingan F1 dan FPR
models = ['LSTM', 'GRU', 'RF']
f1_vals  = [0.9976, 0.9975, 0.9992]
fpr_vals = [0.0046, 0.0028, 0.0015]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
clrs = ['#9C27B0', '#FF9800', '#4CAF50']

# F1
bars = ax1.bar(models, f1_vals, color=clrs, width=0.5)
ax1.set_ylim(0.996, 1.0)
ax1.set_ylabel('F1-Score (DDoS)')
ax1.set_title('F1-Score (lebih tinggi lebih baik)', fontweight='bold')
for bar, v in zip(bars, f1_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
             f'{v:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# FPR
bars2 = ax2.bar(models, [v*100 for v in fpr_vals], color=clrs, width=0.5)
ax2.set_ylabel('FPR (%)')
ax2.set_title('False Positive Rate (lebih rendah lebih baik)', fontweight='bold')
for bar, v in zip(bars2, fpr_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v*100 + 0.01,
             f'{v*100:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Perbandingan Performa Model pada Test Set CIC-DDoS2019', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Demo selesai!')